# 05 — Phase 1b: Round 1 Evaluation — LLM Judge

Development notebook: 3 exercises per version (9 total). The judge now uses Luna and the original Rahimi Table 1 anchors. Only parsed NARRATIVE + DATA + TASK sections are sent, with grade context; teacher notes and condition labels are excluded.

**September 23 revision:** historical source and outputs are preserved in [the archived notebook](../experiments/results/validation/20260923_pf_judge_luna_v1/legacy_05_pf_evaluation_round1.ipynb). Cleared outputs below must not be confused with a new run. Running this entire notebook regenerates exercises; use `scripts/run_pf_judge.py` for a saved-input-only check with prompt/response provenance.

These are automated ratings, not a replication of the paper's blinded human-rating procedure. The historical generator does not record explicit prior instruction; missing evidence may yield a null rating, never zero.

## 1. Setup

In [ ]:
import sys
from pathlib import Path
import importlib
import re

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.utils.llm_client import LLMClient
import src.generation.pf_generator as pfgen
from src.generation.pf_generator import PFGenerator

from scripts.prior_methods.common import load_client

# Preserve the generation path; use Luna for the revised judge.
gen_client = LLMClient.from_role("structured")
judge_client, judge_model = load_client("luna")

gen = PFGenerator(gen_client)
print(f"Generator: {gen_client.model}")
print(f"Judge:     {judge_model}")
print("Ready ✓")

## 2. Generate 3 exercises per version (9 total)

In [ ]:
N_PER_VERSION = 3
exercises = []

for i in range(N_PER_VERSION):
    for version, generate_fn in [("V1", gen.generate_v1), ("V2", gen.generate_v2), ("V3", gen.generate_v3)]:
        ex = generate_fn(grade="7th grade", topic="ratios & proportional relationships", temperature=0.8)
        ex.version = version  # ensure correct label
        exercises.append(ex)
        print(f"[{version}] #{i+1}: valid={ex.is_valid()} | {ex.sections.get('TOPIC', 'N/A')[:50]}")

print(f"\nTotal: {len(exercises)} exercises ({N_PER_VERSION} × 3 versions)")


## 3. Shared paper-aligned rubric and student-only input

In [ ]:
from src.evaluation.pf_rubric import DIMENSIONS, judge_input, legacy_student_text, load_prompt
from scripts.run_pf_judge import rate_case

JUDGE_SYSTEM = load_prompt()["system"]

def judge_input_for(ex):
    return judge_input(legacy_student_text(ex.sections), {"grade": ex.grade})

print(JUDGE_SYSTEM)
print("\n=== ACTUAL JUDGE INPUT ===")
print(judge_input_for(exercises[0]))

## 4. Evaluate exercises with Luna (condition-blinded; GPT-5 omits temperature)

In [ ]:
import random

# Shuffle to prevent order effects
eval_order = list(range(len(exercises)))
random.Random(20260923).shuffle(eval_order)

results = []
for idx in eval_order:
    ex = exercises[idx]
    exercise_text = judge_input_for(ex)
    result = rate_case(judge_client, judge_model, JUDGE_SYSTEM, exercise_text)
    scores = result["scores"] or dict.fromkeys((*DIMENSIONS, "TOTAL"))

    results.append({
        "version": ex.version,
        "exercise_index": idx,
        "judge_input": exercise_text,
        **result,
        **scores,
    })
    print(f"[{len(results)}/{len(exercises)}] {ex.version} | {result['status']} | {scores}")
    if "error" in result:
        print(result["error"])

import pandas as pd
df = pd.DataFrame(results)
print("\nDone ✓")

## 5. Results — per-version means

In [ ]:
# Per-version means
score_columns = [*DIMENSIONS, "TOTAL"]
df[score_columns] = df[score_columns].astype(float)
print("Rating availability (all attempted exercises):")
print(df.groupby(["version", "status"]).size().to_string())
print("Nonmissing ratings per dimension:")
print(df.groupby("version")[score_columns].count().to_string())
summary = df.groupby("version")[score_columns].mean().round(1)
print(summary.to_string())

print("\n--- Per-dimension diff vs V1 ---")
v1_means = df[df["version"]=="V1"][["APK","SSC","AE","MR","OE","TOTAL"]].mean()
for v in ["V2", "V3"]:
    vx_means = df[df["version"]==v][["APK","SSC","AE","MR","OE","TOTAL"]].mean()
    diff = vx_means - v1_means
    print(f"{v} - V1: {dict(diff.round(1))}")

print("\n--- Raw scores ---")
print(df[["version", "status", *score_columns]].to_string(index=False))

## Historical observation (not results for the revised prompt)

The archived execution shows 6/9 exercises receiving the maximum 15/15 total and 9/9 receiving the maximum SSC rating. This warns of limited discrimination, but is not a calibrated quality test. The older prose claim of 8/9 does not match that saved execution. Historical hypotheses were:

1. No calibration baseline — judge sees one exercise at a time, no reference distribution
2. "Expert" frame primes cheerleading, not criticism
3. Structure is conflated with quality — all exercises have proper formatting, so the judge assumes content is good
4. 3-point scale too coarse without concrete anchors
5. No counterfactual reasoning — judge can't ask "what would make this a 2?"

**Literature-backed fix — G-Eval (Liu et al., 2023):**

G-Eval motivates evidence-based evaluation, but asking for analysis before a score does not itself calibrate this PF judge. No agreement with independent human ratings has been measured here.

Judge biases discussed in the literature remain relevant; independent human-rated PF cases are needed to assess this implementation.

**Current implementation:** shared original Table 1 anchors, brief evidence, validated JSON scores, code-computed total, and explicit missing/error states. See [PF_PIPELINE.md](../docs/pf-generation/PF_PIPELINE.md) for the saved-input Luna smoke check and its limits.

### References

- Liu, Y., Iter, D., Xu, Y., Wang, S., Xu, R., & Zhu, C. (2023). G-Eval: NLG Evaluation using GPT-4 with Better Human Alignment. *EMNLP 2023*.
- Zheng, L., Chiang, W.-L., Sheng, Y., et al. (2023). Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena. *NeurIPS 2023 Datasets and Benchmarks Track*.
- Wang, J., Liang, Y., Meng, F., et al. (2023). Is ChatGPT a Good NLG Evaluator? A Preliminary Study. *NewSumm@EMNLP 2023*.